# Training Metrics and Hardware Utilization Visualization

This notebook loads `metrics.json` and creates interactive Plotly charts to analyze model convergence and hardware utilization of your RTX 4070 GPU during training.

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load the metrics data
with open("metrics.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df

## 1. Machine Learning Metrics (Loss, Perplexity, Accuracy)

In [ ]:
# Plot Train Loss and Dev Accuracy
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['train_loss'], name="Train Loss", mode="lines+markers", line=dict(color="royalblue", width=3)),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['val_acc'] * 100, name="Validation Accuracy", mode="lines+markers", line=dict(color="forestgreen", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="Training Loss and Validation Accuracy Over Epochs",
    xaxis_title="Epoch",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)

fig.update_yaxes(title_text="Cross Entropy Loss", secondary_y=False)
fig.update_yaxes(title_text="Accuracy (%)", secondary_y=True)

fig.show()

## 2. Hardware Utilization (Peak VRAM Allocated vs. Reserved)

VRAM allocation tracking is crucial in LLM engineering. 
- **Allocated Memory**: The VRAM actively holding tensors.
- **Reserved Memory**: The VRAM cached by PyTorch's memory allocator (caching allocator) to avoid the high overhead of repeatedly querying the CUDA driver.

In [ ]:
fig_vram = go.Figure()
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_allocated_mb'],
    name='Peak VRAM Allocated (MB)',
    marker_color='crimson'
))
fig_vram.add_trace(go.Bar(
    x=df['epoch'], 
    y=df['peak_vram_reserved_mb'],
    name='Peak VRAM Reserved (MB)',
    marker_color='lightcoral'
))

fig_vram.update_layout(
    barmode='group',
    title_text='Peak GPU VRAM Usage (Allocated vs. Reserved)',
    xaxis_title='Epoch',
    yaxis_title='VRAM (MB)',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_vram.show()

## 3. Data Processing Rate (Throughput vs. Goodput)

- **Throughput**: Total raw tokens (characters) processed per second.
- **Goodput**: Useful tokens processed per second (excluding padding). Since this character-level implementation uses fixed chunking without padding tokens, Goodput is equal to Throughput. However, in variable-length batching, Goodput will be lower due to padding overhead.

In [ ]:
fig_tp = go.Figure()
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['throughput_tokens_sec'],
    name='Throughput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='darkorange', width=3)
))
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['goodput_tokens_sec'],
    name='Goodput (Tokens/Sec)',
    mode='lines+markers',
    line=dict(color='orange', width=2, dash='dash')
))

fig_tp.update_layout(
    title_text='Data Processing Rate (Throughput vs. Goodput)',
    xaxis_title='Epoch',
    yaxis_title='Tokens / Second',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_tp.show()

## 4. Compute Performance (TFLOPs and Model FLOPs Utilization)

Model FLOPs Utilization (MFU) measures how close we are to the theoretical hardware limit of the RTX 4070 (Peak = 121.3 TFLOPs/sec in FP16 tensor computation).

In [ ]:
fig_flops = make_subplots(specs=[[{"secondary_y": True}]])

fig_flops.add_trace(
    go.Scatter(x=df['epoch'], y=df['tflops_per_sec'], name="Achieved TFLOPs/sec", mode="lines+markers", line=dict(color="mediumpurple", width=3)),
    secondary_y=False,
)

fig_flops.add_trace(
    go.Scatter(x=df['epoch'], y=df['mfu_percent'], name="MFU (%)", mode="lines+markers", line=dict(color="hotpink", width=3)),
    secondary_y=True,
)

fig_flops.update_layout(
    title_text="Achieved Compute Performance and Model FLOPs Utilization (MFU)",
    xaxis_title="Epoch",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)

fig_flops.update_yaxes(title_text="Performance (TFLOPs/sec)", secondary_y=False)
fig_flops.update_yaxes(title_text="MFU (%)", secondary_y=True)

fig_flops.show()